In [5]:
%%writefile vector_add.cu
#include <iostream>
#include <cuda_runtime.h>

using namespace std;

// CUDA Kernel
__global__ void vectorAdd(int *A, int *B, int *C, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if(i < n) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    int n;

    cout << "Enter size of vectors: ";
    cin >> n;

    size_t size = n * sizeof(int);

    // Host arrays
    int *h_A = new int[n];
    int *h_B = new int[n];
    int *h_C = new int[n];

    cout << "Enter elements of vector A:\n";
    for(int i = 0; i < n; i++) {
        cin >> h_A[i];
    }

    cout << "Enter elements of vector B:\n";
    for(int i = 0; i < n; i++) {
        cin >> h_B[i];
    }

    // Device pointers
    int *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    // Copy data from Host to Device
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // Define block and grid size
    int blockSize = 256;
    int gridSize = (n + blockSize - 1) / blockSize;

    // Kernel Launch
    vectorAdd<<<gridSize, blockSize>>>(d_A, d_B, d_C, n);

    // Copy result back to Host
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    // Display result
    cout << "\nResultant Vector (A + B):\n";
    for(int i = 0; i < n; i++) {
        cout << h_C[i] << " ";
    }
    cout << endl;

    // Free memory
    delete[] h_A;
    delete[] h_B;
    delete[] h_C;

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}


Overwriting vector_add.cu


In [6]:
!nvcc vector_add.cu -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [7]:
!./vector_add


Enter size of vectors: 5
Enter elements of vector A:
2 2 2 2 2
Enter elements of vector B:
1 1 1 1 1

Resultant Vector (A + B):
3 3 3 3 3 
